## Create Stimuli List

Create the stimuli lists from the final cue-target pairs and the words that have not been normed. This file assumes a 200 word rating set and will randomly split them into chunks. 

### Libraries and Functions

In [1]:
import os
import re
import random
import pandas as pd

TASKS = ["aoa", "image", "concrete", "valence", "arousal", "familiar"]


def clean(s):
    return (
        s.dropna()
        .astype(str)
        .str.strip()
        .replace("", pd.NA)
        .dropna()
    )


def load_word_pool(lang, ml_one_network_root=".."):
    """Real-word cue/target pool for `lang`, sourced from the finalized
    trial list (01-Translation/05_final_languages/<lang>/<lang>_trials_final.csv)
    -- the same file the priming build/deploy/processing pipeline already
    uses -- rather than a separately hand-maintained word-pairs file, so
    there's one current word source instead of two that can drift apart."""
    trials_path = os.path.join(
        ml_one_network_root, "01-Translation", "05_final_languages",
        lang, f"{lang}_trials_final.csv",
    )
    trials = pd.read_csv(trials_path)

    real = trials[(trials["cue_type"] == "word") & (trials["target_type"] == "word")]
    cue_col, target_col = f"{lang}_cue", f"{lang}_target"

    words = pd.concat([clean(real[cue_col]), clean(real[target_col])], ignore_index=True)
    return words.drop_duplicates().tolist()


def load_variables_normed(lang, variables_normed_path=None, output_base="."):
    """Loads <lang>_variables_normed.csv, produced by <lang>.Rmd's
    category_missing(): per that Rmd's own doc comment, TRUE means no
    external source covers this word for this category yet (still needs
    norming); FALSE means some source already has a value for it. So
    `== True` is the "needed" filter below, not `== False`."""
    path = variables_normed_path or os.path.join(output_base, lang, f"{lang}_variables_normed.csv")
    normed = pd.read_csv(path)

    if "word" not in normed.columns:
        raise ValueError("variables_normed must contain 'word' column")

    normed["word"] = clean(normed["word"])
    for task in TASKS:
        normed[task] = normed[task].astype(str).str.lower().map(
            {"true": True, "false": False, "1": True, "0": False}
        )
    return normed.drop_duplicates(subset="word")


def make_buckets(words, candidate_words, bucket_size, rng):
    words = list(words)
    rng.shuffle(words)

    buckets = []
    used = set()

    for i in range(0, len(words), bucket_size):
        chunk = words[i:i + bucket_size]
        used.update(chunk)

        # pad short final chunks with unused candidate words
        if len(chunk) < bucket_size:
            needed_n = bucket_size - len(chunk)
            fillers = [w for w in candidate_words if w not in used]
            rng.shuffle(fillers)
            chunk.extend(fillers[:needed_n])
            used.update(chunk)

        buckets.append(chunk)

    return buckets


def existing_bucketed_words(lang, task, output_base="."):
    """Words already sitting in a list_N.csv on disk, whether or not that
    list has ever been launched/collected. Used only to tell "never queued
    anywhere" apart from "queued but not launched yet, e.g. a partial
    final list held back on purpose" -- a word with data (in
    completion_summary) is handled separately via list_open/still_needed."""
    task_dir = os.path.join(output_base, lang, task)
    words = set()
    if os.path.isdir(task_dir):
        for fn in os.listdir(task_dir):
            if re.match(rf"^{task}_list_\d+\.csv$", fn):
                words.update(clean(pd.read_csv(os.path.join(task_dir, fn))["word"]))
    return words


def find_topupable_list(lang, task, output_base, bucket_size):
    """The last (highest-numbered) list_N.csv for this task, if it's short
    of bucket_size AND has zero collected data anywhere -- strong evidence
    it was generated but never actually launched, so it's still safe to
    edit directly rather than leaving it partial and spinning up a new
    list for the remainder. Returns (n, path, existing_words) or None."""
    task_dir = os.path.join(output_base, lang, task)
    if not os.path.isdir(task_dir):
        return None

    nums = []
    for fn in os.listdir(task_dir):
        m = re.match(rf"^{task}_list_(\d+)\.csv$", fn)
        if m:
            nums.append(int(m.group(1)))
    if not nums:
        return None

    last_n = max(nums)
    path = os.path.join(task_dir, f"{task}_list_{last_n}.csv")
    existing_words = clean(pd.read_csv(path)["word"]).tolist()

    if len(existing_words) >= bucket_size:
        return None

    return last_n, path, existing_words


def next_list_number(lang, task, output_base="."):
    task_dir = os.path.join(output_base, lang, task)
    nums = []
    if os.path.isdir(task_dir):
        for fn in os.listdir(task_dir):
            m = re.match(rf"^{task}_list_(\d+)\.csv$", fn)
            if m:
                nums.append(int(m.group(1)))
    return max(nums, default=0) + 1


def build_norming_lists(
    lang,
    variables_normed_path=None,
    output_base=".",
    ml_one_network_root="..",
    bucket_size=200,
    seed=42,
):
    """One-time bootstrap pass: every word variables_normed.csv marks as
    not-yet-normed (False) for a task, chunked into list_N.csv files
    starting at 1. Run this once per language, before any collection has
    started -- see build_update_lists() for later passes."""
    rng = random.Random(seed)

    candidate_words = load_word_pool(lang, ml_one_network_root)
    candidate_set = set(candidate_words)
    normed = load_variables_normed(lang, variables_normed_path, output_base)

    task_word_lists = {}
    for task in TASKS:
        needed = normed.loc[normed[task] == True, "word"].tolist()
        needed = [w for w in needed if w in candidate_set]
        task_word_lists[task] = list(dict.fromkeys(needed))

    task_buckets = {
        task: make_buckets(task_word_lists[task], candidate_words, bucket_size, rng)
        for task in TASKS
    }

    for task, buckets in task_buckets.items():
        task_dir = os.path.join(output_base, lang, task)
        os.makedirs(task_dir, exist_ok=True)
        for i, bucket in enumerate(buckets, start=1):
            pd.DataFrame({"word": bucket}).to_csv(
                os.path.join(task_dir, f"{task}_list_{i}.csv"), index=False
            )

    print(f"Seed: {seed}")
    print(f"Candidate pool: {len(candidate_words)}")
    for task in TASKS:
        sizes = [len(b) for b in task_buckets[task]]
        print(f"{task}: needed={len(task_word_lists[task])}, buckets={sizes}")

    return task_buckets


def build_update_lists(
    lang,
    completion_summary_path=None,
    variables_normed_path=None,
    output_base=".",
    ml_one_network_root="..",
    bucket_size=200,
    default_expected_responses=35,
    top_up_partial=True,
    seed=42,
):
    """Update pass, run after some lists have been collected. Source of
    truth shifts from build_norming_lists' bootstrap logic (just
    variables_normed) to the real completion summary spaml2-private pushes
    (pipeline/sync_public_repo.R's push_completion_summary()):

      - variables_normed still gates eligibility -- a word already normed
        by another study never comes back, same as the bootstrap pass.
      - completion_summary supplies each eligible word's deficit
        (expected - recorded responses). A word absent from the summary
        (never collected at all, e.g. its list hasn't been launched yet)
        is treated as a full deficit, same as the bootstrap pass -- so
        "brand new" and "needs a top-up" are just different deficit sizes
        on the same scale, not two separate cases.
      - completion_summary's list_open column (from studylist.csv's
        download flag) gates whether a deficit is actually actionable:
        a word still sitting in a list that's actively collecting is
        skipped here -- it needs more participants on its CURRENT list,
        not a new duplicate one. Only words in a CLOSED list (or not
        queued anywhere at all) get bucketed.

    Words are bucketed by DEFICIT SIZE (largest first), not randomly -- a
    bucket's recruit requirement is driven by its worst-off word, so
    grouping similar deficits together (rather than mixing a brand-new
    word with one that's nearly done) keeps each new list's recruit count
    as low as possible. This matches how aoa_makeup_list_1/2 were split by
    hand (see spaml2-private/pipeline/list_requirements_uk.md). Numbering
    continues from whatever's already on disk instead of restarting at 1.
    """
    rng = random.Random(seed)

    candidate_words = load_word_pool(lang, ml_one_network_root)
    candidate_set = set(candidate_words)
    normed = load_variables_normed(lang, variables_normed_path, output_base)

    summary_path = completion_summary_path or os.path.join(output_base, lang, f"{lang}_completion_summary.csv")
    if os.path.exists(summary_path):
        completion = pd.read_csv(summary_path)
        if "list_open" not in completion.columns:
            completion["list_open"] = False
    else:
        print(f"No completion summary found at {summary_path} -- treating every eligible word as a full deficit.")
        completion = pd.DataFrame(columns=["task", "item", "recorded_responses", "expected_responses", "list_open"])

    task_word_lists = {}
    task_buckets = {}

    for task in TASKS:
        eligible = set(normed.loc[normed[task] == True, "word"]) & candidate_set

        task_completion = completion.loc[completion["task"] == task].set_index("item")

        already_queued = existing_bucketed_words(lang, task, output_base)

        deficits = {}
        for word in eligible:
            if word in task_completion.index:
                row = task_completion.loc[word]
                # Still sitting in a list that's actively collecting
                # (studylist.csv download == TRUE for at least one list
                # holding it) -- skip it here, it needs more participants
                # on its CURRENT list, not a new duplicate one.
                if bool(row["list_open"]):
                    continue
                deficit = max(int(row["expected_responses"]) - int(row["recorded_responses"]), 0)
            elif word in already_queued:
                # No collected data anywhere, but it's already sitting in
                # an on-disk list that just hasn't been launched yet (e.g.
                # a partial final list held back on purpose) -- it needs to
                # be deployed, not duplicated into a new one.
                continue
            else:
                deficit = default_expected_responses
            if deficit > 0:
                deficits[word] = deficit

        # Shuffle first (random tiebreak among equal deficits), then sort
        # descending by deficit -- Python's sort is stable so the shuffle
        # order survives within each deficit tier.
        ordered = list(deficits.keys())
        rng.shuffle(ordered)
        ordered.sort(key=lambda w: deficits[w], reverse=True)

        # Top up an existing unlaunched partial list (e.g. the last, short
        # list from a previous bootstrap/update pass) before spinning up
        # any new list numbers, so a held-back partial list gets filled
        # rather than left short forever.
        topup = find_topupable_list(lang, task, output_base, bucket_size) if top_up_partial else None
        if topup:
            topup_n, topup_path, topup_existing = topup
            already_has_data = any(w in task_completion.index for w in topup_existing)
            if already_has_data:
                topup = None  # not actually unlaunched -- leave it alone

        if topup:
            topup_n, topup_path, topup_existing = topup
            slot = bucket_size - len(topup_existing)
            added = ordered[:slot]
            ordered = ordered[slot:]
            if added:
                pd.DataFrame({"word": topup_existing + added}).to_csv(topup_path, index=False)
                print(f"  topped up {task}_list_{topup_n}.csv: {len(topup_existing)} -> {len(topup_existing) + len(added)}")

        task_word_lists[task] = ordered

        buckets = []
        for i in range(0, len(ordered), bucket_size):
            chunk = ordered[i:i + bucket_size]
            if len(chunk) < bucket_size:
                used = set(chunk)
                fillers = [w for w in ordered if w not in used]
                chunk = chunk + fillers[: bucket_size - len(chunk)]
            buckets.append(chunk)

        task_buckets[task] = buckets

    for task, buckets in task_buckets.items():
        if not buckets:
            continue
        task_dir = os.path.join(output_base, lang, task)
        os.makedirs(task_dir, exist_ok=True)
        start_n = next_list_number(lang, task, output_base)
        for offset, bucket in enumerate(buckets):
            n = start_n + offset
            pd.DataFrame({"word": bucket}).to_csv(
                os.path.join(task_dir, f"{task}_list_{n}.csv"), index=False
            )

    print(f"Candidate pool: {len(candidate_words)}")
    for task in TASKS:
        sizes = [len(b) for b in task_buckets[task]]
        print(f"{task}: still needed={len(task_word_lists[task])}, new buckets={sizes}")

    return task_buckets


### Build Stimuli Lists

In [2]:
LANG = "uk"  # change this to run for a different language

task_buckets = build_norming_lists(
    lang=LANG,
    output_base=".",
    ml_one_network_root="..",
    bucket_size=200,
    seed=4536,
)


Seed: 4536
Candidate pool: 1939
aoa: needed=1939, buckets=[200, 200, 200, 200, 200, 200, 200, 200, 200, 139]
image: needed=1939, buckets=[200, 200, 200, 200, 200, 200, 200, 200, 200, 139]
concrete: needed=1939, buckets=[200, 200, 200, 200, 200, 200, 200, 200, 200, 139]
valence: needed=1939, buckets=[200, 200, 200, 200, 200, 200, 200, 200, 200, 139]
arousal: needed=1939, buckets=[200, 200, 200, 200, 200, 200, 200, 200, 200, 139]
familiar: needed=1939, buckets=[200, 200, 200, 200, 200, 200, 200, 200, 200, 139]


### Update Stimuli Lists (after collection has started)

Run this instead of the bootstrap cell above once some lists have already
been deployed and collected. It reads the completion summary
`spaml2-private` pushes via `pipeline/sync_public_repo.R`'s
`push_completion_summary()`, so it only generates new lists for words that
are eligible (per `variables_normed.csv`) and not already done or queued --
run `push_completion_summary(lang)` there first to refresh it.


In [ ]:
LANG = "uk"  # change this to run for a different language

task_buckets = build_update_lists(
    lang=LANG,
    output_base=".",
    ml_one_network_root="..",
    bucket_size=200,
    seed=4536,
)
